# 00 — Radar Intuition and Baseline Parameters

This notebook explains what pulse radar is, why the quiet listening interval matters, and why the baseline radar parameters were chosen.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/active-radar-tracker-basics/blob/radar-tracker-notebooks/beginner/00-radar-intuition.ipynb)

Use this link if you want to open the notebook in Colab and follow along in a browser notebook environment.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Colab starts without this repository on disk, so clone it before importing the beginner helpers.
REPO_URL = "https://github.com/vinculum3141-ship-it/active-radar-tracker-basics.git"
BRANCH_NAME = "radar-tracker-notebooks"
REPO_DIR = Path("/content/active-radar-tracker-basics")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH_NAME, "--single-branch", REPO_URL, str(REPO_DIR)],
            check=True,
        )
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)
    print(f"Cloned the repository to {REPO_DIR}")
else:
    print("Running locally; the repository is already available in this workspace.")

## What this notebook teaches

Pulse radar is a system that transmits a short burst of energy, then stops transmitting and listens for the echo. That simple idea drives everything that follows in this course.

By the end of this notebook, the learner should be able to explain:

- what pulse radar is,
- why the radar listens after it transmits,
- how duty cycle is computed,
- and how a range delay turns into a range estimate.

## Setup and baseline values

Before we calculate anything, we load the small beginner helper package and create one named baseline radar specification. If you are running this notebook in Colab, run the bootstrap cell above first so the repository is cloned and the helper package is available.

In [ ]:
from beginner.helpers import BaselineRadarSpec, baseline_spec, duty_cycle, delay_samples_for_range, range_from_delay_samples, wavelength_m
from beginner.helpers.plotting import apply_notebook_style

# Set the shared notebook styling once for consistent plots.
apply_notebook_style()

# Create one named radar specification so the lesson reads naturally.
radar_spec = BaselineRadarSpec()
radar_spec

## Baseline radar reference

The baseline radar specification is the fixed example we use throughout the beginner lessons. Each number has a job to do, and this table explains what each one controls.

| Quantity | Value | What it controls |
|---|---:|---|
| Carrier frequency | 2.45 GHz | Connects the radar to wavelength and later Doppler calculations |
| Bandwidth | 5 MHz | Sets the range resolution we can achieve with pulse compression |
| Pulse width | 20 microseconds | Controls how long each burst is transmitted |
| PRI | 1 millisecond | Sets how long the radar has to listen before the next pulse |
| Sampling rate | 20 MHz | Tells us how many samples we record per second |
| Target range | 1000 m | Gives us one concrete delay-to-range example |

## What pulse radar means

A pulse radar does not transmit continuously. It sends a short burst of energy and then waits. During the transmit burst, the antenna is busy sending power out. During the quiet time after the burst, the radar listens for the echo that comes back from a target.

This back-and-forth rhythm is the foundation of range measurement:

- the transmitted pulse creates the event,
- the quiet listening window lets the echo arrive,
- the time delay of that echo tells us how far away the target is.

The important idea is that the radar is not trying to hear while it is speaking. It has to stop transmitting before it can listen clearly.

## Why the radar listens after it transmits

The radar needs a quiet interval because the echo from a distant target takes time to come back. If the radar kept transmitting continuously, the weak echo would be buried under the outgoing signal.

That is why we define a pulse repetition interval, or PRI. The pulse width is the short transmit burst, and the rest of the PRI is the listening window.

The key equations are:

$$
D = \frac{\tau}{T}
$$

for duty cycle, and

$$
R = \frac{c \cdot \tau_{delay}}{2}
$$

for range from round-trip delay.

In this notebook, we use the baseline values to compute those quantities explicitly so the learner sees the arithmetic before the helper functions are introduced.

In [ ]:
# Compute the lesson quantities step by step with the equations written out in the notebook.
duty_cycle_value = radar_spec.pulse_width_s / radar_spec.pri_s
round_trip_delay_seconds = (2.0 * radar_spec.target_range_m) / 299_792_458.0
round_trip_delay_samples = round(round_trip_delay_seconds * radar_spec.fs_hz)
estimated_range_m = (round_trip_delay_samples / radar_spec.fs_hz) * 299_792_458.0 / 2.0
carrier_wavelength_m = 299_792_458.0 / radar_spec.fc_hz

print("Baseline radar values used in this lesson:")
print(f"  carrier frequency = {radar_spec.fc_hz / 1e9:.2f} GHz")
print(f"  bandwidth = {radar_spec.bandwidth_hz / 1e6:.1f} MHz")
print(f"  pulse width = {radar_spec.pulse_width_s * 1e6:.1f} microseconds")
print(f"  PRI = {radar_spec.pri_s * 1e3:.1f} milliseconds")
print(f"  sampling rate = {radar_spec.fs_hz / 1e6:.1f} MHz")
print()
print("Step 1: duty cycle = pulse width / PRI")
print(f"  duty cycle = {radar_spec.pulse_width_s:.2e} / {radar_spec.pri_s:.2e} = {duty_cycle_value:.3f}")
print()
print("Step 2: round-trip delay = 2 * range / speed of light")
print(f"  delay = 2 * {radar_spec.target_range_m:.1f} / 299,792,458 = {round_trip_delay_seconds:.3e} s")
print(f"  delay = {round_trip_delay_samples} samples at {radar_spec.fs_hz / 1e6:.1f} MHz")
print()
print("Step 3: convert delay back to range")
print(f"  estimated range = {estimated_range_m:.1f} m")
print()
print("Step 4: carrier wavelength")
print(f"  wavelength = {carrier_wavelength_m:.4f} m")

{
    "duty_cycle": duty_cycle_value,
    "delay_seconds": round_trip_delay_seconds,
    "delay_samples": round_trip_delay_samples,
    "range_m": estimated_range_m,
    "wavelength_m": carrier_wavelength_m,
}

## Checkpoint

In your own words, why does a pulse radar need a listening window after the transmit pulse?

Then answer this: if the pulse width stays the same but the PRI becomes longer, what happens to the duty cycle?

## Common mistake

A longer PRI does not make the radar transmit more often. It actually gives the radar more time to listen, so the duty cycle gets smaller. That is why a pulse radar can be active for only a small fraction of the time but still gather useful range information.

Another common mistake is to treat the delay as a technical detail instead of the main measurement. In pulse radar, that delay is the signal that tells us where the target is.

In [ ]:
# Now use the helper functions and the shared baseline spec to show the same answers more compactly.
helper_radar_spec = baseline_spec()
helper_duty_cycle_value = duty_cycle(helper_radar_spec.pulse_width_s, helper_radar_spec.pri_s)
helper_round_trip_delay_samples = delay_samples_for_range(helper_radar_spec.target_range_m, helper_radar_spec.fs_hz)
helper_estimated_range_m = range_from_delay_samples(helper_round_trip_delay_samples, helper_radar_spec.fs_hz)
helper_carrier_wavelength_m = wavelength_m(helper_radar_spec.fc_hz)

print("Helper-based version of the same calculations:")
print(f"  duty cycle = {helper_duty_cycle_value:.3f}")
print(f"  delay samples = {helper_round_trip_delay_samples}")
print(f"  estimated range = {helper_estimated_range_m:.1f} m")
print(f"  wavelength = {helper_carrier_wavelength_m:.4f} m")

{
    "duty_cycle": helper_duty_cycle_value,
    "delay_samples": helper_round_trip_delay_samples,
    "range_m": helper_estimated_range_m,
    "wavelength_m": helper_carrier_wavelength_m,
}

## Why the helpers exist

The calculation above shows the equations directly so the learner can see the arithmetic happen. After that first pass, we move the same logic into the helper package so the later notebooks can reuse it without repeating the derivation.

That is what the helper functions are for:

- they keep the same baseline formulas available in later notebooks,
- they reduce repeated code,
- and they make later lesson cells shorter once the learner already understands the idea.

Going forward, use the helpers when you want the lesson to stay readable, but keep the first occurrence of a concept visible in the notebook itself. Once the learner has seen the equation work by hand, the helper version is the clean way to reuse the same idea.

In [ ]:
# Draw a simple timing picture that shows when the radar transmits and when it listens.
fig, ax = plt.subplots(figsize=(10, 2.5))

ax.broken_barh([(0, radar_spec.pulse_width_s)], (0.25, 0.35), facecolors="#d95f02", label="Transmit pulse")
ax.broken_barh([(radar_spec.pulse_width_s, radar_spec.pri_s - radar_spec.pulse_width_s)], (0.25, 0.35), facecolors="#1b9e77", label="Listening window")

ax.text(radar_spec.pulse_width_s / 2, 0.7, "Transmit", ha="center", va="bottom")
ax.text(radar_spec.pulse_width_s + (radar_spec.pri_s - radar_spec.pulse_width_s) / 2, 0.7, "Listen", ha="center", va="bottom")

ax.set_xlim(0, radar_spec.pri_s)
ax.set_ylim(0, 1)
ax.set_xlabel("Time within one PRI (seconds)")
ax.set_yticks([])
ax.set_title("Pulse radar transmits briefly, then listens for the echo")
ax.legend(loc="upper right")

plt.show()

## Summary

In this notebook, the learner met pulse radar as a simple but powerful idea: transmit a short burst, wait quietly, and use the returning echo to measure range. The lesson showed that the listening window is not an optional extra; it is the part of the radar cycle that makes the measurement possible.

The notebook also introduced the baseline radar example used throughout the beginner track. That shared example gives the course a common set of radar values so each later notebook can build on the same physical situation. From here forward, the learner should think of those values as the reference case that connects the radar story across range, timing, Doppler, and the later application notebooks.

The main takeaway is that radar is not just about transmitting energy. It is about carefully timing the transmit-and-listen cycle so the system can turn a delay into a physical measurement with meaning.